In [1]:
import glob
import json
import os
import re
from xml.etree import ElementTree as ET
# from src.data.xml_extraction import gen_xml_paths

In [3]:
field_keywords = json.load(open("external/field_keywords.json", encoding="utf8"))

In [4]:
def gen_xml_paths(path: str | os.PathLike) -> list[str]:
    """
    Collect xmls from a path
    Retries needed as this was originally on a network path that failed occasionally
    :param path: str : A location of transkribus model output xmls
    :return: list[str], list[str]
    """

    attempts = 0
    while attempts < 3:
        xmls = glob.glob(path)
        if xmls:
            break
        else:
            attempts += 1
            continue
    else:
        raise IOError(f"Failed to connect to {path}")

    return xmls

In [5]:
def parse_custom_attribute_string(element: ET.Element) -> list[tuple[str, tuple[str, str]]]:
    """
    Parse the custom attributes of an XML element
    Convert the custom string into a list of (Transkribus) tags and tag values

    Args:
        element (Element): _description_

    Returns:
        list[tuple[str, tuple[str, str]]]: _description_
    """
    attributes_raw = element.attrib.get("custom")
    # Handle misformatted Unicode, U+0020 (space), U+0027 (apostrophe)
    attributes = attributes_raw.replace(r"\u0020", " ").replace(r"\u0027", "'")
    attrib_pair_re = re.compile(r"(?P<tag>\w+) (?P<text>\{[\.\w\s:;\d\\'’-]+\})")
    attrib_inner_re = re.compile(r"(?P<tag>\w+):(?P<text>[\.\w\s\d\\'’-]+)")
    all_attribs = attrib_pair_re.findall(attributes)

    inner_found = [(k, attrib_inner_re.findall(v[1:-1])) for k,v in all_attribs]
    # breakpoint()
    return inner_found

In [8]:
xmls = gen_xml_paths("raw/*.xml")

In [9]:
tree = ET.parse(xmls[1])

In [10]:
root = tree.getroot()

In [11]:
[print(parse_custom_attribute_string(e)) for e in root[1][2:]]

[('readingOrder', [('index', '0')]), ('structure', [('type', 'Provenance')])]
[('readingOrder', [('index', '1')]), ('structure', [('type', 'Binding')])]
[('readingOrder', [('index', '2')]), ('structure', [('type', 'Provenance')])]
[('readingOrder', [('index', '3')]), ('structure', [('type', 'Dating')])]
[('readingOrder', [('index', '4')]), ('structure', [('type', 'Dating')])]
[('readingOrder', [('index', '5')]), ('structure', [('type', 'Dating')])]
[('readingOrder', [('index', '6')]), ('structure', [('type', 'table')])]


[None, None, None, None, None, None, None]

In [12]:
field_keywords

{'Binding': ['Binding', 'Bound', 'Rebound', 'Inlaid'],
 'Dating': ['Dating'],
 'Provenance': ['Provenance',
  'Presented',
  'From',
  'Grenville Copy',
  'King George III’s Copy',
  'Bought']}

In [13]:
field_keywords.get('Dating', None)

['Dating']

- iterate over all xmls
- iterate over all regions in each xml
- get the structure type for that region
- check if region is one of binding, dating, provenance
- if yes check if region starts with one of the field keywords
- if yes assign to good list
- if no assign to bad list

- iterate over regions in one xml
- get the structure type for that region
- check if region is one of binding, dating, provenance
- if yes check get keywords for field
- get all text for region
- process text to remove blank lines
- check region starts with one of the field keywords
- create good list/bad list
- assign to good/list bad list
- generalise to iterate over all xmls

In [14]:
[print(parse_custom_attribute_string(e)) for e in root[1][2:]]

[('readingOrder', [('index', '0')]), ('structure', [('type', 'Provenance')])]
[('readingOrder', [('index', '1')]), ('structure', [('type', 'Binding')])]
[('readingOrder', [('index', '2')]), ('structure', [('type', 'Provenance')])]
[('readingOrder', [('index', '3')]), ('structure', [('type', 'Dating')])]
[('readingOrder', [('index', '4')]), ('structure', [('type', 'Dating')])]
[('readingOrder', [('index', '5')]), ('structure', [('type', 'Dating')])]
[('readingOrder', [('index', '6')]), ('structure', [('type', 'table')])]


[None, None, None, None, None, None, None]

In [31]:
good_regions = []
bad_regions = []
# line below is iterating over the xml
# for each iteration lets us access each text region
for region in root[1][2:]:
    # line below getting the custom attribute string and structuring it and assigning it to attributes
    attributes = parse_custom_attribute_string(region)
    # line below is printing field type for each text region
    # print(attributes[1][1][0][1])
    # line below assigns field type to region_field_type
    region_field_type = attributes[1][1][0][1]
    # line below assigns field names for OCR that we want to add to MARC records to target_field_types
    target_field_types = ['Binding', 'Dating', 'Provenance']
    # looks for target_field_types in the region_field_type
    if region_field_type in target_field_types:
        # assigns field key words to target_field_keywords
        target_field_keywords = field_keywords.get(region_field_type)
        # creates list to gather text lines for region
        region_lines = []
        # iterating over the text lines in each text region
        for line in region[1:-1]:
            # getting the text
            line_text = line[2][0].text
            # adding the text for the line to the list of region lines
            region_lines.append(line_text)

        non_blank_lines = []
        for line in region_lines:
            if line:
                non_blank_lines.append(line)
        for line in non_blank_lines[:4]:
            kw_present = [kw in line for kw in target_field_keywords]
            if any(kw_present):
                good_regions.append([region_field_type, region])
                break
        else:
            breakpoint()
            bad_regions.append([region_field_type, region])

print(f"Good regions: {good_regions}")
print(f"Bad regions: {bad_regions}")

> /tmp/ipython-input-2771419626.py(38)<cell line: 0>()
     36         else:
     37             breakpoint()
---> 38             bad_regions.append([region_field_type, region])
     39 
     40 print(f"Good regions: {good_regions}")

ipdb> non_blank_lines
['Type 7: 95G. but with Type 3B: 64G. as commentary type, with', 'see below, p. 296), is represented from this, with the same text', 'unsigned leaf (bound first) perhaps belonging to quire e.', 'signed aij and containing the beginning of the text, and an', 'this fragment, which consists of two leaves, the first (bound last)', 'No more seems to be known of the edition represented by', 'Picardus commentary', 'used as exemplar one of the French editions with the Odo', 'who does not specify commentaries. It seems likely that Pynson', 'manuscripts used in English schools see Bonaventure (1961),', 'The present edition is the earliest printed in England. For', 'who was taught by Odo. See Quinn (1971, pp. 407-8).', 'Louis, Duke of Orleans, b

In [27]:
def print_bad_region_text(text_region: ET.Element) -> None:
    """
    Print the text of a bad region
    First extract all text lines from text region then
    print the text line by line
    :param text_region: ET.Element : A text region in a transkribus model output xmls
    :return: None
    """
    # creates list to gather text lines for region
    region_lines = []
    # iterating over the text lines in each text region
    for line in region[1:-1]:
        # getting the text
        line_text = line[2][0].text
        # adding the text for the line to the list of region lines
        region_lines.append(line_text)
    # print all lines in text region
    for line in region_lines:
        print(line)
    print("\n\n")

    return None

In [32]:
for field_type, region in bad_regions:
    print(f"Bad region for {field_type}")
    print_bad_region_text(region)


Bad region for Dating
Type 7: 95G. but with Type 3B: 64G. as commentary type, with
see below, p. 296), is represented from this, with the same text
unsigned leaf (bound first) perhaps belonging to quire e.
signed aij and containing the beginning of the text, and an
this fragment, which consists of two leaves, the first (bound last)
No more seems to be known of the edition represented by
Picardus commentary
used as exemplar one of the French editions with the Odo
who does not specify commentaries. It seems likely that Pynson
manuscripts used in English schools see Bonaventure (1961),
The present edition is the earliest printed in England. For
who was taught by Odo. See Quinn (1971, pp. 407-8).
Louis, Duke of Orleans, brother of Charles VI, for the Dauphin
Eudes, or Oudart de Fouilly), written in 1406-7 at the request of
commentary is that by Magister Odo Picardus (also known as
2800). In most of the French editions, as in those of Pynson, the
Auctores octo, with or without commentary (s

In [24]:
name = "Jeanette"
print(f"hello {name}")

hello Jeanette
